In [1]:
import requests
import pandas as pd
from utils import clean_text_cols, clean_datetime_cols

In [2]:
# --- 1. CONFIGURATION ---
URL = "https://gamma-api.polymarket.com/events"
# Note: Polymarket 'active' and 'closed' are strings in their API
PARAMS = {"active": "true", "closed": "false", "limit": 1000}

In [3]:
# --- 2. FETCH & INITIAL LOAD ---
response = requests.get(URL, params=PARAMS)
events_data = response.json()
raw_poly_df = pd.DataFrame(events_data)


In [4]:
# --- 3. TRANSFORM: TAGS ---
# Handle nested tags first
tags_exploded = raw_poly_df[['id', 'tags']].explode('tags').dropna()
tag_labels = pd.json_normalize(tags_exploded['tags'])
tag_labels['event_id'] = tags_exploded['id'].values

# Group labels into lists and join them into a string for clean_text_cols to handle
grouped_tags = tag_labels.groupby('event_id')['label'].apply(lambda x: ', '.join(x)).reset_index()
grouped_tags.columns = ['event_id', 'tags']


In [5]:
# --- 4. TRANSFORM: EVENTS ---
event_cols = {
    'id': 'event_id',
    'title': 'event_title',
    'description': 'description',
    'startDate': 'start_date'
}

poly_events = raw_poly_df[list(event_cols.keys())].rename(columns=event_cols)
poly_events = pd.merge(poly_events, grouped_tags, on='event_id', how='left')

In [6]:
# --- 5. TRANSFORM: MARKETS ---
markets_exploded = raw_poly_df[['id', 'markets']].explode('markets').dropna()
markets_df = pd.json_normalize(markets_exploded['markets'])

# Map schema to match your Kalshi master format
market_map = {
    'id': 'market_id',
    'question': 'market_subtitle',
    'bestBid': 'yes_bid',
    'bestAsk': 'yes_ask',
    'endDateIso': 'expiration'
}

# Add event_id back for the join
markets_df['event_id'] = markets_exploded['id'].values
markets_df = markets_df.rename(columns=market_map)

# Keep only the columns we need
final_market_cols = [c for c in market_map.values() if c in markets_df.columns] + ['event_id']
markets_df = markets_df[final_market_cols].copy()


In [7]:
# --- 6. FINAL MERGE & GLOBAL CLEANING ---
master_market_df = pd.merge(markets_df, poly_events, on='event_id', how='left')
master_market_df = clean_text_cols(master_market_df)

date_cols = ['start_date', 'expiration']
master_market_df = clean_datetime_cols(master_market_df, date_cols=date_cols)

# Final formatting
master_market_df = master_market_df.sort_values(['event_id', 'market_id']).reset_index(drop=True)
master_market_df['platform'] = 'poly'
master_market_df.to_csv('data/markets/poly_markets.csv', index=False)

/Users/blakeuribe/Desktop/quasibets_testing/utils.py:67: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')


In [8]:
event_cols = ['event_id', 'event_title', 'description', 'start_date', 'tags', 'platform']

existing_cols = [c for c in event_cols if c in master_market_df.columns]
poly_events_df = master_market_df[existing_cols].drop_duplicates(subset=['event_id']).copy()

poly_events_df = clean_text_cols(poly_events_df)
poly_events_df.to_csv('data/events/poly_events.csv', index=False)
poly_events_df

,event_id,event_title,description,start_date,tags,platform
0,16167,microstrategy sells any bitcoin by,this market will resolve to yes if microstrate...,NaN,finance economy business 2025 predictions cryp...,poly
4,16183,kraken ipo by,this market will resolve to yes if kraken us b...,NaN,exchange tech crypto finance business 2025 pre...,poly
8,16263,macron out by,this market will resolve to yes if emmanuel ma...,NaN,france politics macron world 2025 predictions ...,poly
11,16423,uk election called by,this is a market on predicting the date when t...,NaN,starmer uk pedophile england,poly
15,17526,china x india military clash by,this is a market on the likelihood of a milita...,NaN,india politics china geopolitics world,poly
...,...,...,...,...,...,...
7103,83930,uk designate the irgc a terrorist organization...,this market will resolve to yes if the united ...,NaN,politics world,poly
7104,84018,next ceo of apple,this market will resolve according to the firs...,NaN,big tech business apple tech,poly
7126,84066,gmgn launch a token by,this market will resolve to yes if gmgn https ...,NaN,crypto pre market gmgn token launch,poly
7129,84543,southern miss golden eagles vs prairie view a ...,this event is for the cbb game between souther...,NaN,sports basketball ncaa games ncaa basketball,poly
